# 🏋️ Exercise Detector - GPU Training on Colab

Trains an **ST-GCN + BiLSTM** model for **exercise classification** (number of classes from config) + **form quality assessment** on Colab's free T4 GPU.

- **Model**: ST-GCN (4 blocks) + Angle branch MLP + BiLSTM → ~6.88M parameters
- **Input**: YOLOv8 pose keypoints (17 COCO joints) + 12 joint angles
- **Expected speedup**: ~50-100x vs CPU (seconds/batch instead of minutes)

**Before starting:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 1: Load Project from Google Drive

Upload `exercise detector.zip` to your Google Drive, then run the cell below.

The zip contains the full project folder. We only extract what's needed for training (~120 MB of skeleton data + code + checkpoint).

In [ ]:
from google.colab import drive
import os, zipfile, shutil

# Mount Google Drive
drive.mount('/content/drive')

# Path to your zip in Google Drive
zip_path = '/content/drive/MyDrive/exercise detector.zip'

if not os.path.exists(zip_path):
    print(f"❌ File not found at: {zip_path}")
    print("Make sure 'exercise detector.zip' is in your Google Drive root folder.")
    print("If it's in a subfolder, update zip_path above.")
else:
    size_gb = os.path.getsize(zip_path) / 1e9
    print(f"Found zip: {size_gb:.2f} GB")
    print("Extracting only training-essential files (skipping raw videos)...")

    # Only extract files we need from exercise detector/test2/
    needed_prefixes = [
        'exercise detector/test2/configs/',
        'exercise detector/test2/checkpoints/',
        'exercise detector/test2/data/processed/skeletons/',
        'exercise detector/test2/data/splits/',
        'exercise detector/test2/src/',
        'exercise detector/test2/scripts/',
    ]

    work_dir = '/content/exercise_detector'
    os.makedirs(work_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        extracted = 0
        for member in z.namelist():
            if any(member.startswith(p) for p in needed_prefixes):
                # Strip the "exercise detector/test2/" prefix
                rel_path = member.replace('exercise detector/test2/', '', 1)
                if not rel_path or rel_path.endswith('/'):
                    continue  # skip directories

                out_path = os.path.join(work_dir, rel_path)
                os.makedirs(os.path.dirname(out_path), exist_ok=True)

                with z.open(member) as src, open(out_path, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                extracted += 1

    os.chdir(work_dir)
    print(f"\n✅ Extracted {extracted} files to {work_dir}")
    !ls -la
    print()
    !du -sh data/processed/skeletons/ checkpoints/ src/ configs/ data/splits/ scripts/

In [ ]:
!pip install -q torch torchvision pyyaml pandas numpy tqdm tensorboard

## Step 2: Verify Data

Load the training dataset and verify that skeleton data, angles, and labels are all present and correctly shaped.

In [ ]:
import yaml
import sys
sys.path.insert(0, '.')

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

from src.data.dataset import SkeletonDataset
ds = SkeletonDataset(
    skeleton_dir='data/processed/skeletons',
    labels_file='data/splits/train.csv',
    window_size=30, stride=15,
    config=config,
)
print(f"Train windows: {len(ds)}")
forms = [w['form'] for w in ds.windows]
print(f"  Correct: {sum(f==1 for f in forms)} ({100*sum(f==1 for f in forms)/len(forms):.1f}%)")
print(f"  Incorrect: {sum(f==0 for f in forms)} ({100*sum(f==0 for f in forms)/len(forms):.1f}%)")

sample = ds[0]
print(f"\nSample shapes:")
print(f"  skeleton: {sample['skeleton'].shape}")  # (30, 17, 6)
print(f"  angles:   {sample['angles'].shape}")     # (30, 12)
print(f"  exercise: {sample['exercise']}")
print(f"  form:     {sample['form']}")

## Step 3: Train!

This cell replicates the full training pipeline from `scripts/train.py`:
- Loads config and creates dataloaders with augmentation + class-balanced sampling
- Builds the ST-GCN + BiLSTM model
- Resumes from `checkpoints/last.pt` if it exists
- Uses uncertainty-weighted multi-task loss with class weights + label smoothing
- Warmup + cosine annealing LR schedule
- Early stopping with patience
- Saves `best.pt` and `last.pt` checkpoints

In [ ]:
import os, sys, yaml, torch
import numpy as np
from collections import Counter
sys.path.insert(0, '.')

from src.data.dataset import create_dataloaders
from src.models.classifier import build_model
from src.training.trainer import Trainer
from src.training.losses import MultiTaskLoss

# Load config
with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create dataloaders
loaders = create_dataloaders(
    skeleton_dir='data/processed/skeletons',
    splits_dir='data/splits',
    window_size=config['data']['window_size'],
    stride=config['data']['stride'],
    batch_size=config['training']['batch_size'],
    num_workers=2,
    config=config,
)

train_loader = loaders['train']
val_loader = loaders.get('val', train_loader)
print(f"Train samples: {len(train_loader.dataset)}")
if 'val' in loaders:
    print(f"Val samples: {len(val_loader.dataset)}")

# Build model
model = build_model(config)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")

# Class weights for exercise head (inverse frequency)
train_cfg = config['training']
exercise_class_weights = None
if train_cfg.get('exercise_class_weights', False):
    class_counts, _ = train_loader.dataset.get_class_counts()
    num_classes = config['model']['num_exercises']
    total = sum(class_counts.values())
    weights = torch.zeros(num_classes)
    for cls_idx, count in class_counts.items():
        weights[cls_idx] = total / (num_classes * count)
    exercise_class_weights = weights.to(device)
    print(f"Using exercise class weights (min={weights.min():.2f}, max={weights.max():.2f})")

# Loss function
label_smoothing = train_cfg.get('label_smoothing', 0.0)
loss_fn = MultiTaskLoss(
    exercise_class_weights=exercise_class_weights,
    label_smoothing=label_smoothing,
)
print(f"Label smoothing: {label_smoothing}")

# Optimizer (includes loss_fn learnable params for uncertainty weighting)
all_params = list(model.parameters()) + list(loss_fn.parameters())
optimizer = torch.optim.Adam(
    all_params,
    lr=train_cfg['learning_rate'],
    weight_decay=train_cfg.get('weight_decay', 0.0001),
)

# Scheduler: warmup + cosine annealing
warmup_epochs = train_cfg.get('warmup_epochs', 0)
total_epochs = train_cfg['epochs']

if warmup_epochs > 0:
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs,
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_epochs - warmup_epochs, eta_min=train_cfg['learning_rate'] * 0.01,
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs],
    )
    print(f"Scheduler: {warmup_epochs}-epoch linear warmup + cosine annealing")
else:
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_epochs, eta_min=train_cfg['learning_rate'] * 0.01,
    )

# Resume from checkpoint
start_epoch = 0
best_val_loss = float('inf')
resume_path = 'checkpoints/last.pt'
if os.path.exists(resume_path):
    print(f"\nResuming from checkpoint: {resume_path}")
    checkpoint = torch.load(resume_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    loss_fn.load_state_dict(checkpoint['loss_fn_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint.get('epoch', 0)
    best_val_loss = checkpoint.get('val_loss', float('inf'))
    print(f"Resuming from epoch {start_epoch}, best val loss: {best_val_loss:.4f}")
else:
    print("\nStarting training from scratch")

# Create trainer
trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    config=config,
    checkpoint_dir='checkpoints',
    log_dir='logs',
)

if start_epoch > 0:
    trainer.best_val_loss = best_val_loss

# Train!
patience = train_cfg.get('early_stopping_patience', 25)
print(f"\nTraining: epochs {start_epoch+1} → {total_epochs}, patience={patience}")
print("=" * 60)

trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=total_epochs,
    start_epoch=start_epoch,
)

In [ ]:
# Run this anytime to backup checkpoints to Google Drive (safe from session crashes)
import shutil, torch

drive_backup = '/content/drive/MyDrive/exercise_detector_checkpoints'
os.makedirs(drive_backup, exist_ok=True)

for ckpt in ['best.pt', 'last.pt']:
    src = f'checkpoints/{ckpt}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_backup, ckpt))
        c = torch.load(src, map_location='cpu')
        print(f"✅ {ckpt} → Drive  (epoch {c.get('epoch','?')}, val_loss={c.get('val_loss','?'):.4f})")

## Step 4: Evaluate

Load the best checkpoint and evaluate on the test set. Compare accuracy against the previous CPU baseline of 52.96%.

In [ ]:
from src.data.dataset import create_dataloaders
import torch

# Load best checkpoint
checkpoint = torch.load('checkpoints/best.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint (epoch {checkpoint.get('epoch', '?')})")

# Get test loader
test_loader = loaders.get('test')
if test_loader is None:
    loaders_eval = create_dataloaders(
        skeleton_dir='data/processed/skeletons',
        splits_dir='data/splits',
        window_size=30, stride=30,
        batch_size=32,
        config=config,
    )
    test_loader = loaders_eval['test']

correct = 0
total = 0
from collections import Counter
predictions = []

with torch.no_grad():
    for batch in test_loader:
        skeleton = batch['skeleton'].to(device)
        angles = batch['angles'].to(device)
        exercise = batch['exercise'].to(device)

        ex_logits, form_logits = model(skeleton, angles)
        preds = ex_logits.argmax(1)
        correct += (preds == exercise).sum().item()
        total += exercise.size(0)
        predictions.extend(preds.cpu().numpy().tolist())

accuracy = 100 * correct / total
print(f"\nTest Accuracy: {accuracy:.2f}% ({correct}/{total})")
print(f"Previous accuracy was 52.96% \u2014 {'improved! \U0001f389' if accuracy > 52.96 else 'needs more training'}")

## Step 5: Save & Download Trained Model

Saves trained checkpoints to Google Drive (so you don't lose them if Colab disconnects) and also offers a direct download.

In [ ]:
import shutil

# 1. Save to Google Drive (survives session disconnects)
drive_save_dir = '/content/drive/MyDrive/exercise_detector_checkpoints'
os.makedirs(drive_save_dir, exist_ok=True)

for ckpt in ['best.pt', 'last.pt']:
    src = f'checkpoints/{ckpt}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_save_dir, ckpt))
        print(f"✅ Saved {ckpt} to Google Drive ({os.path.getsize(src)/1e6:.1f} MB)")

print(f"\nCheckpoints saved to: {drive_save_dir}")
print("Copy best.pt and last.pt into your local checkpoints/ folder to use with the app.")

# 2. Also offer direct download
from google.colab import files
shutil.make_archive('/content/trained_checkpoints', 'zip', '.', 'checkpoints')
files.download('/content/trained_checkpoints.zip')